# HARP-PCB v0.2 — Auto-placement in Google Colab

Upload an **unrouted** KiCad `.kicad_pcb`, even if footprints are already partially placed or overlap. The placer **restarts** placement, generates a new rectangular outline, and saves intermediate KiCad and image snapshots when requested.

**Important limitations:** Not a router, not an electrical/physical DRC, not a verified footprint library. Synthetic land patterns and ambiguous external-facing connectors **block placement by default**. You may explicitly opt in to a *geometry-only experiment*; it is not a manufacturing-ready board. Boards with copper tracks/vias/zones are refused to prevent accidental loss or invalidation. Nonrectangular existing outlines require explicit consent to replace.

**No LLM is needed** for physical placement. Optional IntelCAD Claude is used only for an aggregate, non-authoritative **report explanation**. Never paste an API key into a notebook output or GitHub issue.

## 1 · Settings (run this first)

- Set `SHOW_STAGES = True` for `.kicad_pcb` and PNG snapshots of the initial legal placement, local refinement and final result.
- `ALLOW_UNVERIFIED_FOOTPRINTS` defaults to `False`; **do not set it to True for manufacturing**.
- `ALLOW_UNRESOLVED_ORIENTATION` defaults to `False`; describe the side and local opening direction in the edge-rule cell below.
- `USE_LLM_REPORT = False` means no API key is requested and nothing is sent to an external service.

In [ ]:
# Colab / placement configuration
SHOW_STAGES = True
SNAPSHOT_EVERY = 2000  # 0 disables additional annealing checkpoints
ITERATIONS = 4000
SEED = 42
CLEARANCE_MM = 0.65
EDGE_MARGIN_MM = 2.0
ALLOW_UNVERIFIED_FOOTPRINTS = False
ALLOW_UNRESOLVED_ORIENTATION = False
ALLOW_REPLACE_NONRECTANGULAR_OUTLINE = False

# LLM is OPTIONAL and does not edit the PCB.
USE_LLM_REPORT = False
MODEL = "cc/claude-haiku-5-5"  # alternatively "cc/claude-opus-5-5"
BASE_URL = "https://ai-router.intelcad.ai/v1"
AI_API_KEY = None
if USE_LLM_REPORT:
    from getpass import getpass
    AI_API_KEY = getpass("IntelCAD API key (masked; kept in notebook memory): ").strip()
    if not AI_API_KEY:
        raise ValueError("AI report enabled but no API key was supplied")
print("LLM is", "enabled for aggregate report only" if USE_LLM_REPORT else "disabled (not required)")

## 2 · Clone the public repository and install dependencies

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path("/content/PCB_placement")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/amirhossein-alishahi/PCB_placement.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[preview]"], check=True)
subprocess.run([sys.executable, "-m", "harp_pcb", "--help"], check=True, stdout=subprocess.DEVNULL)
print("Installed from:", REPO)

## 3 · Upload your `.kicad_pcb` and inspect preflight

The first report lists **synthetic land patterns**, **access-direction candidates** and any explicit or marker-inferred directions. This is important for SD, USB, edge-actuated switches and connectors: an ordinary footprint name **does not prove which side faces outward**.

In [ ]:
from google.colab import files
import json, shutil
from pathlib import Path
uploaded = files.upload()
boards = [name for name in uploaded if name.lower().endswith('.kicad_pcb')]
if len(boards) != 1:
    raise ValueError("Upload exactly one .kicad_pcb file")
WORK = Path('/content/harp_job')
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = WORK/'input.kicad_pcb'
shutil.copyfile(Path(boards[0]),SOURCE)
print('Selected:', boards[0], '| bytes:', SOURCE.stat().st_size)

## 4 · Set hard mechanical-facing rules (when needed)

This config maps **Reference → side on final board + outward-facing direction in the unrotated footprint**.

For example `"J1": {"side":"right", "outward_local":"right"}` means the connector opening at unrotated local **+X** must face the **right-hand board edge**. `"side":"any"` permits any edge but still forces the opening to face outward. With KiCad board coordinates at angle 0, right=(+X), left=(-X), top=(-Y), bottom=(+Y). Angles 90/180/270 rotate the footprint accordingly. For the local opening orientation, rely on the footprint datasheet/3D model — don't guess it from the name.

The engine may infer `outward_local` if a footprint contains a suitably positioned **`PCB Edge` marker** on `Dwgs.User`/`Cmts.User`/Fab. In other cases explicit rules are recommended. Any ambiguous access component blocks by default.

In [ ]:
EDGE_RULES = {
    # "J1": {"side": "right", "outward_local": "right", "edge_offset_mm": 0.5},
    # "J2": {"side": "bottom", "outward_local": "right", "edge_offset_mm": 0.5},
    # "SW1": {"side": "any", "outward_local": "top", "edge_offset_mm": 0.5},
}
RULE_FILE = WORK/'edge_rules.json'
RULE_FILE.write_text(json.dumps({"components": EDGE_RULES}, indent=2), encoding='utf-8')
PREFLIGHT_FILE = WORK/'preflight.json'
pre = subprocess.run([sys.executable,'-m','harp_pcb',str(SOURCE),'--direction-rules',str(RULE_FILE),
                      '--preflight-only','--report',str(PREFLIGHT_FILE)],
                     capture_output=True, text=True)
print(pre.stdout if pre.returncode == 0 else pre.stderr)
if pre.returncode:
    raise RuntimeError('Preflight failed — inspect input board and edge rules')
PREFLIGHT = json.loads(PREFLIGHT_FILE.read_text())
print('Synthetic:',len(PREFLIGHT['synthetic_footprints']),
      '| Unresolved access-direction:',PREFLIGHT['unresolved_directions'])

## 5 · Auto-place and validate

If this cell refuses to run, **read the error first**: replace synthetic/dummy footprints with verified KiCad land patterns, add missing edge rules, or explicitly opt into a **geometry-only** experiment in cell 1. Do not delete the `SYNTHETIC` silkscreen warning as a substitute for fixing the geometry.

In [ ]:
OUTPUT = WORK/'placed.kicad_pcb'
REPORT = WORK/'placement_report.json'
PREVIEW = WORK/'placed_preview.png'
STAGES = WORK/'stages'
cmd = [sys.executable,'-m','harp_pcb',str(SOURCE),'-o',str(OUTPUT),
       '--iterations',str(ITERATIONS),'--seed',str(SEED),
       '--clearance',str(CLEARANCE_MM),'--edge-margin',str(EDGE_MARGIN_MM),
       '--direction-rules',str(RULE_FILE),
       '--report',str(REPORT),'--preview',str(PREVIEW)]
if SHOW_STAGES:
    cmd += ['--stages-dir',str(STAGES),'--snapshot-every',str(SNAPSHOT_EVERY)]
if ALLOW_UNVERIFIED_FOOTPRINTS:
    cmd.append('--allow-unverified-footprints')
if ALLOW_UNRESOLVED_ORIENTATION:
    cmd.append('--allow-unresolved-orientation')
if ALLOW_REPLACE_NONRECTANGULAR_OUTLINE:
    cmd.append('--replace-nonrectangular-outline')
run = subprocess.run(cmd, capture_output=True, text=True)
print(run.stdout[-6000:] if run.returncode == 0 else run.stderr[-4000:])
if run.returncode:
    raise RuntimeError('Placement rejected. Fix the reported issue or adjust explicit flags/rules.')
RESULT = json.loads(REPORT.read_text())
print('Output:', OUTPUT, 'size:', OUTPUT.stat().st_size, 'bytes')

## 6 · View final and intermediate stages

In [ ]:
from IPython.display import display, Image, Markdown
if PREVIEW.exists():
    display(Image(filename=str(PREVIEW)))
if SHOW_STAGES and (STAGES/'manifest.json').exists():
    manifest = json.loads((STAGES/'manifest.json').read_text())
    print('Saved stages:', [m['name'] for m in manifest])
    import ipywidgets as widgets
    stage_choice = widgets.Dropdown(options=[(f"{i+1}: {m['name']}", i) for i,m in enumerate(manifest)],
                                    description='Stage:')
    out = widgets.Output()
    def change_stage(change=None):
        with out:
            out.clear_output(wait=True)
            chosen=manifest[stage_choice.value]
            display(Markdown(f"**{chosen['name']}** · `{chosen['pcb']}`"))
            display(Image(filename=str(STAGES/chosen['preview'])))
    stage_choice.observe(change_stage, names='value')
    display(stage_choice,out)
    change_stage()

## 7 · Optional Claude review (aggregate results only)

This **does not change** placement. Only numerical counts, timing and high-level warnings are transmitted when `USE_LLM_REPORT = True`. For circuit design confidentiality, raw `.kicad_pcb`, netlists, part references and footprint names are **not sent**.

In [ ]:
if USE_LLM_REPORT:
    import urllib.request, urllib.error
    aggregate = {k: RESULT.get(k) for k in (
        'footprints','pads','nets','board_width_mm','board_height_mm','board_area_mm2',
        'min_bounding_box_gap_mm','runtime_seconds','accepted_moves')}
    aggregate['synthetic_count'] = len(RESULT['footprint_audit']['synthetic_footprints'])
    aggregate['unresolved_orientation_count'] = len(RESULT['footprint_audit']['unresolved_directions'])
    messages = [{'role':'system','content': 'Explain a PCB geometry-only placement report. Do not claim manufacturability, KiCad DRC, correct orientation, or routability. Reply in Persian.'},
                {'role':'user','content': json.dumps(aggregate)}]
    body=json.dumps({'model':MODEL,'messages':messages,'temperature':0.1,'max_tokens':700}).encode()
    req=urllib.request.Request(BASE_URL.rstrip('/')+'/chat/completions',data=body,method='POST',
              headers={'Authorization':'Bearer '+AI_API_KEY,'Content-Type':'application/json'})
    try:
        with urllib.request.urlopen(req,timeout=75) as res:
            answer=json.load(res)
        print(answer['choices'][0]['message']['content'])
    except (urllib.error.HTTPError,urllib.error.URLError,KeyError,ValueError) as exc:
        print('AI report unavailable. Placement is unaffected:',type(exc).__name__)
else:
    print('Skipped: no LLM is required for auto-placement.')

## 8 · Download results

`placed.kicad_pcb` is a **geometry-only** output; run KiCad DRC and review external-facing openings and actual footprint datasheets before routing or manufacturing.

In [ ]:
from google.colab import files
import zipfile
ARCHIVE = WORK/'harp_results.zip'
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in [OUTPUT,REPORT,PREVIEW,PREFLIGHT_FILE,RULE_FILE]:
        if file.exists():
            z.write(file,arcname=file.name)
    if SHOW_STAGES and STAGES.exists():
        for file in STAGES.rglob('*'):
            if file.is_file():
                z.write(file,arcname='stages/'+str(file.relative_to(STAGES)))
print('Archive:', ARCHIVE,'size:',ARCHIVE.stat().st_size,'bytes')
files.download(str(ARCHIVE))